In [ ]:
# ffmpeg -i /home/aistudio/models/realityscan/Video/shitang.mp4 -r 15 -vf "scale='min(1024,iw)':'min(1024,ih)':force_original_aspect_ratio=decrease" -qscale:v 1 /home/aistudio/models/realityscan/Video/shitangframes/%04d.jpg

In [8]:
import pycolmap
from pathlib import Path

# 1. Setup Absolute Paths
# Based on your root /home/aistudio

image_path =  Path("/home/aistudio/models/realityscan/Video/shitangframes")
output_path =  Path("/home/aistudio/models/ColmapOutputs/Shitang")
output_path.mkdir(parents=True, exist_ok=True)

database_folder = output_path / "database"
database_folder.mkdir(exist_ok=True)
database_path = database_folder / "database.db"

# Create output directories

sparse_path = output_path / "sparse"
sparse_path.mkdir(exist_ok=True)

reader_options = pycolmap.ImageReaderOptions()


sift_options = pycolmap.SiftExtractionOptions()

sift_options.max_num_features = 8192
# 2. Feature Extraction
# This identifies key points in your 400 images
print(f"Extracting features from: {image_path}")
pycolmap.extract_features(
    database_path=str(database_path),
    image_path=str(image_path),

)

Extracting features from: /home/aistudio/models/realityscan/Video/shitangframes


I20260331 11:52:59.155519 140386526426880 feature_extraction.cc:502] === Feature extraction ===
I20260331 11:52:59.157555 140386551604992 sift.cc:754] Creating SIFT GPU feature extractor
I20260331 11:52:59.256212 140386518034176 feature_extraction.cc:280] Processed file [1/437]
I20260331 11:52:59.256274 140386518034176 feature_extraction.cc:283]   Name:            .ipynb_checkpoints/0003-checkpoint.jpg
I20260331 11:52:59.256278 140386518034176 feature_extraction.cc:292]   Dimensions:      1600 x 900
I20260331 11:52:59.256282 140386518034176 feature_extraction.cc:295]   Camera:          #1 - SIMPLE_RADIAL
I20260331 11:52:59.256286 140386518034176 feature_extraction.cc:298]   Focal Length:    1920.00px
I20260331 11:52:59.256297 140386518034176 feature_extraction.cc:302]   Features:        20359 (SIFT)
I20260331 11:52:59.299530 140386518034176 feature_extraction.cc:280] Processed file [2/437]
I20260331 11:52:59.299576 140386518034176 feature_extraction.cc:283]   Name:            .ipynb_ch

In [9]:



# 3. Feature Matching
# Exhaustive matching compares every image to every other image.
# For 400 images, this is thorough and reliable for 3D reconstruction.
print("Matching features...")
pycolmap.match_exhaustive(database_path=database_path)




# 4. Reconstruction (Structure from Motion)
# This calculates the 3D points and camera positions.
print("Starting incremental reconstruction...")
reconstructions = pycolmap.incremental_mapping(
    database_path=str(database_path),
    image_path=str(image_path),
    output_path=sparse_path
)






# 5. Finalize for Gaussian Splatting
# Gaussian Splatting usually expects the '0' folder (points3D.bin, cameras.bin, images.bin)
if reconstructions:
    print(f"Success! Found {len(reconstructions)} reconstruction clusters.")
    # Export the primary model to the '0' subfolder
    final_output = sparse_path / "0"
    final_output.mkdir(exist_ok=True)
    reconstructions[0].write(str(final_output))
    print(f"Model saved to: {final_output}")
else:
    print("Reconstruction failed. Check if your images have enough overlap.")

Matching features...


I20260331 11:56:03.559908 140386526426880 feature_matching.cc:195] === Feature matching & geometric verification ===
I20260331 11:56:03.560493 140386518034176 feature_matching_utils.cc:80] Bind FeatureMatcherWorker to GPU device 0
I20260331 11:56:03.561458 140386518034176 sift.cc:1547] Creating SIFT GPU feature matcher
I20260331 11:56:03.563831 140386526426880 pairing.cc:180] Generating exhaustive image pairs...
I20260331 11:56:03.563850 140386526426880 pairing.cc:213] Processing block [1/9, 1/9]
I20260331 11:56:17.769099 140386526426880 feature_matching.cc:217] in 14.205s
I20260331 11:56:17.769171 140386526426880 pairing.cc:213] Processing block [1/9, 2/9]
I20260331 11:56:28.355836 140386526426880 feature_matching.cc:217] in 10.587s
I20260331 11:56:28.355906 140386526426880 pairing.cc:213] Processing block [1/9, 3/9]
I20260331 11:56:43.338535 140386526426880 feature_matching.cc:217] in 14.983s
I20260331 11:56:43.338623 140386526426880 pairing.cc:213] Processing block [1/9, 4/9]
I20260

Starting incremental reconstruction...


I20260331 12:12:54.540186 140391789279040 database_cache.cc:139]  95266 in 0.740s
I20260331 12:12:54.540255 140391789279040 database_cache.cc:147] Loading images...
I20260331 12:12:54.789015 140391789279040 database_cache.cc:241]  437 in 0.249s (connected 437, loaded 437)
I20260331 12:12:54.789125 140391789279040 database_cache.cc:255] Loading pose priors...
I20260331 12:12:54.789143 140391789279040 database_cache.cc:263]  0 in 0.000s
I20260331 12:12:54.789148 140391789279040 database_cache.cc:271] Building correspondence graph...
I20260331 12:13:18.441883 140391789279040 database_cache.cc:300]  in 23.653s (ignored 0)
I20260331 12:13:18.449744 140391789279040 timer.cc:90] Elapsed time: 0.411 [minutes]
I20260331 12:13:18.640909 140391789279040 incremental_pipeline.cc:377] Finding good initial image pair
I20260331 12:16:13.729727 140391789279040 incremental_pipeline.cc:401] Registering initial image pair #293 and #19
I20260331 12:16:13.782330 140391789279040 incremental_pipeline.cc:419] 

In [1]:
import pycolmap
import numpy as np

# Path to your COLMAP sparse folder
sparse_folder = "/home/aistudio/models/ColmapOutputs/Shitang/sparse/0"

# Load reconstruction
recon = pycolmap.Reconstruction(sparse_folder)
# recon.cameras is the mapping of CameraID -> Camera object
print(f"Number of cameras found: {len(recon.cameras)}")

for camera_id, camera in recon.cameras.items():
    print(f"\n--- Camera ID: {camera_id} ---")
    print(f"Model: {camera.model_name}")
    print(f"Dimensions: {camera.width} x {camera.height}")
    
    # These are the raw values from cameras.bin
    params = camera.params 
    print(f"Raw Parameters: {params}")

# points3D is a dict: {point3D_id: Point3D_object}
print(f"Total points in sparse cloud: {len(recon.points3D)}")

# Let's look at the first few points
for i, (point_id, point) in enumerate(recon.points3D.items()):
    if i >= 3: break
    
    xyz = point.xyz      # [x, y, z] coordinates
    rgb = point.color    # [r, g, b] (0-255)
    error = point.error  # Reprojection error (lower is better)
    
    print(f"Point {point_id}: Pos={xyz}, Color={rgb}, Error={error:.4f}")

E20260331 12:50:50.230617 140570743682880 reconstruction.cc:974] rigs, cameras, frames, images, points3D files do not exist at "/home/aistudio/models/ColmapOutputs/Shitang/sparse/0"


ValueError: [reconstruction.cc:974] rigs, cameras, frames, images, points3D files do not exist at "/home/aistudio/models/ColmapOutputs/Shitang/sparse/0"